In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import deque
from tqdm import tqdm
import random
from copy import deepcopy
from utils import lineage_name_mapping, load_json

In [ ]:
lineage_data = load_json('./data/cell_lineage.json')
# lineage_exp_df = pd.read_csv("data/protein/aggregated_all/s3.csv", index_col=0)
# lineage_exp_df = pd.read_csv("data/protein/aggregated_all/s3_lineage_umap_embedding_5d.csv", index_col=0).T
# lineage_exp_df = pd.read_csv("data/protein/aggregated_all/s3_pca_5d.csv", index_col=0)
lineage_exp_df = pd.read_csv("data/protein/aggregated_all/s3_pca_10d.csv", index_col=0)
lineage_exp_df = lineage_exp_df.fillna(0)
# binarize lineage_exp_df, making every value > 3.2 as 1, otherwise 0
# lineage_exp_df = (lineage_exp_df > 3.2).astype(int)
tracking_df_all = pd.read_csv("./data/embryo2/tracks.txt", sep="\t")
tracking_time_cutoff = 247
tracking_scale = 0.1625
tracking_df = tracking_df_all.loc[tracking_df_all["t"] <= tracking_time_cutoff]

def depth_cutoff_func(depth):
    if depth <= 9:
        return 1
    else:
        return float('inf')

In [3]:
from collections import defaultdict
terminal_nodes = []
parent_dict = {}
terminal_parents_dict = defaultdict(list)
terminal_ancestry = defaultdict(list)
lineage_name_to_id = {}
def dfs(node, parent, ancestors=[]):
    children = node.get("children", [])
    lookup_name = lineage_name_mapping(node["did"])
    if lookup_name not in lineage_name_to_id:
        lineage_name_to_id[lookup_name] = len(lineage_name_to_id)
    if len(children) == 0:
        p_lookup_name = lineage_name_mapping(parent['did'])
        parent_dict[lookup_name] = p_lookup_name
        terminal_nodes.append(lookup_name)
        terminal_parents_dict[p_lookup_name].append(lookup_name)
        terminal_ancestry[lookup_name] = ancestors
    else:
        if parent is not None:
            p_lookup_name = lineage_name_mapping(parent['did'])
            parent_dict[lookup_name] = p_lookup_name
        for child in children:
            dfs(child, node, ancestors + [lookup_name])

dfs(lineage_data, None)

In [4]:
cell_type_df = pd.read_csv("data/2023-06-29_entropy_cell_key_V2.csv")
count = 0
lineage_type_code_dict = defaultdict(str)
type_code_dict = {}
na_lineages = []
for lineage in terminal_nodes:
    cur_cell_type_df = cell_type_df[cell_type_df['wormweb.lineage'] == lineage]
    if cur_cell_type_df.empty:
        # print(f"No cell types found for lineage: {lineage}")
        na_lineages.append(lineage)
        continue
    cur_lineage_types = cur_cell_type_df["wormweb.type"]
    # remove nan from cur_lineage_types
    cur_lineage_types = cur_lineage_types[~cur_lineage_types.isna()]
    cur_lineage_types = cur_lineage_types.unique()
    if len(cur_lineage_types) == 0:
        # print(f"No cell types found for lineage: {lineage}")
        na_lineages.append(lineage)
        continue
    cur_type = cur_lineage_types[0]
    if cur_type not in type_code_dict:
        # type_code_dict[cur_type] = chr(97 + len(type_code_dict))  # start from 'a'
        type_code_dict[cur_type] = len(type_code_dict)
    lineage_type_code_dict[lineage] = type_code_dict[cur_type]
# dead/NA type
na_type_code = len(type_code_dict)
type_code_dict["dead/NA"] = na_type_code # unknown/dead type
for lineage in na_lineages:
    lineage_type_code_dict[lineage] = na_type_code

In [5]:
id_to_parent_id_list = [-1] * len(lineage_name_to_id)
for child_name, parent_name in parent_dict.items():
    child_id = lineage_name_to_id[child_name]
    parent_id = lineage_name_to_id[parent_name]
    id_to_parent_id_list[child_id] = parent_id
terminal_ids = [lineage_name_to_id[name] for name in terminal_nodes]
lineage_ids = list(range(len(lineage_name_to_id)))

In [6]:
lineage_type_code_by_id_dict = {}
for lineage_name, type_code in lineage_type_code_dict.items():
    lineage_id = lineage_name_to_id[lineage_name]
    lineage_type_code_by_id_dict[lineage_id] = type_code
terminal_type_nums = len(np.unique([lineage_type_code_by_id_dict[lineage_name_to_id[name]] for name in terminal_nodes]))

In [7]:
def complexity_score(lineage_parent_list: list[int]):
    complexity_code_dict = deepcopy(lineage_type_code_by_id_dict)
    cell_div_count = 0
    parent_code_mapping = {}
    bfs_queue = deque(terminal_ids)
    while bfs_queue:
        cur_node_lineage_id = bfs_queue.popleft()
        cur_parent_lineage_id = lineage_parent_list[cur_node_lineage_id] if cur_node_lineage_id != -1 else None
        cur_node_code = complexity_code_dict[cur_node_lineage_id]
        if cur_node_code is None:
            # print(f"Type code not found for node: {cur_node_lineage_id}")
            continue
        if cur_parent_lineage_id is None:
            # print(f"Parent not found for node: {cur_node_lineage_id}")
            continue
        if cur_parent_lineage_id not in complexity_code_dict:
            complexity_code_dict[cur_parent_lineage_id] = cur_node_code
            bfs_queue.append(cur_parent_lineage_id)
        else:
            cur_parent_code = complexity_code_dict[cur_parent_lineage_id]
            if cur_parent_code > cur_node_code:
                parent_code_tuple = (cur_parent_code, cur_node_code)
            else:
                parent_code_tuple = (cur_node_code, cur_parent_code)
            if parent_code_tuple not in parent_code_mapping:
                parent_code_mapping[parent_code_tuple] = len(parent_code_mapping)+terminal_type_nums
            complexity_code_dict[cur_parent_lineage_id] = parent_code_mapping[parent_code_tuple]
            cell_div_count += 1

    return len(parent_code_mapping) / cell_div_count

In [8]:
complexity_score(id_to_parent_id_list)

0.29055912007332724

In [15]:
def genetic_complexity_optimization(lineage_parent_list, iterations=100000):
    
    cur_complexity_score = complexity_score(lineage_parent_list)
    print("Initial Complexity Score:", cur_complexity_score)
    cur_lineage_parent_list = deepcopy(lineage_parent_list)
    for it in tqdm(range(iterations)):
        # randomly swap two nodes
        lineage_id1, lineage_id2 = random.sample(lineage_ids, 2)
        next_lineage_parent_list = deepcopy(cur_lineage_parent_list)
        parent_lineage_id1 = next_lineage_parent_list[lineage_id1]
        parent_lineage_id2 = next_lineage_parent_list[lineage_id2]
        next_lineage_parent_list[lineage_id1] = parent_lineage_id2
        next_lineage_parent_list[lineage_id2] = parent_lineage_id1
        new_complexity_score = complexity_score(next_lineage_parent_list)
        if new_complexity_score < cur_complexity_score:
            cur_complexity_score = new_complexity_score
            cur_lineage_parent_list = next_lineage_parent_list
            print(f"Iteration {it}: New best complexity score: {cur_complexity_score}")

    return cur_lineage_parent_list, cur_complexity_score

In [ ]:
genetic_complexity_optimization(id_to_parent_id_list)